# Bloco 01 -- Leitura e Exploracao de Dados

**Aula 08 | Trilha Databricks | Jornada de Dados**

---

Este e o primeiro notebook de PySpark da trilha. Ate agora voce usou Databricks principalmente via SQL, configurou Unity Catalog, criou pipelines com DLT e modelou dados em Medallion. Agora vamos mergulhar na **PySpark DataFrame API**.

Neste bloco voce vai aprender a:
- Carregar tabelas do catalogo com `spark.table()`
- Ler arquivos em diferentes formatos com `spark.read`
- Definir schemas explicitos com `StructType`
- Explorar DataFrames com `printSchema()`, `describe()`, `display()` e mais
- Comparar DataFrame API com `spark.sql()`

**Pre-requisito:** O script `dataset/script.sql` ja foi executado e as tabelas Northwind existem no catalogo.

## 1. Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    FloatType,
    DateType,
    ShortType,
)

# Altere para o nome do seu catalogo
CATALOG = "northwind"
SCHEMA = "bronze"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Criar Volume para armazenar arquivos (substitui DBFS /tmp)
spark.sql("CREATE VOLUME IF NOT EXISTS demo_files")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/demo_files"
print(f"Volume disponível em: {VOLUME_PATH}")

## 2. Carregando tabelas com `spark.table()`

A forma mais direta de carregar uma tabela que ja existe no Unity Catalog e usando `spark.table()`. Como voce ja executou o `script.sql`, todas as tabelas Northwind estao disponiveis.

`spark.table()` retorna um **DataFrame** -- a estrutura central do PySpark para manipulacao de dados.

In [ ]:
# Carregando tabelas do catalogo
orders = spark.table("orders")
customers = spark.table("customers")

# Verificando o tipo -- e um DataFrame
print(type(orders))

In [ ]:
# Visualizacao rapida com display() -- funcao nativa do Databricks
display(orders)

## 3. Lendo arquivos CSV com `spark.read.csv()`

Nem sempre os dados estao em tabelas do catalogo. Muitas vezes voce recebe arquivos CSV, JSON ou Parquet. O PySpark tem o `spark.read` para lidar com isso.

Para demonstrar, vamos **escrever** a tabela `orders` como CSV e depois **ler de volta**. Assim o exemplo fica auto-contido.

In [ ]:
# Escrevendo orders como CSV para demonstracao
orders.write.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,
    mode="overwrite",
)

In [ ]:
# Lendo o CSV com inferSchema=True
# O Spark tenta "adivinhar" os tipos das colunas
orders_csv = spark.read.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,
    inferSchema=True,
)

orders_csv.printSchema()

> **Observacao:** Compare o schema inferido com o original. O `inferSchema` faz uma passada extra nos dados para detectar tipos, e nem sempre acerta -- por exemplo, pode inferir `integer` quando o tipo correto e `short`, ou `string` quando deveria ser `date`. Em producao, sempre defina o schema explicitamente.

In [ ]:
# Comparando: schema original da tabela vs schema inferido do CSV
print("=== Schema original (spark.table) ===")
orders.printSchema()

print("\n=== Schema inferido (spark.read.csv) ===")
orders_csv.printSchema()

## 4. Schema explicito com `StructType`

Em producao, voce **nunca** deve confiar no `inferSchema`. A forma segura e definir o schema explicitamente usando `StructType` e `StructField`.

Cada `StructField` recebe:
- Nome da coluna (`string`)
- Tipo de dado (`StringType()`, `IntegerType()`, `FloatType()`, `DateType()`, etc.)
- Se aceita nulos (`True` / `False`)

Vamos definir o schema de `order_details` como exemplo.

In [ ]:
# Primeiro, vamos escrever order_details como CSV
order_details = spark.table("order_details")

order_details.write.csv(
    f"{VOLUME_PATH}/order_details_csv",
    header=True,
    mode="overwrite",
)

In [ ]:
# Definindo o schema explicito para order_details
order_details_schema = StructType([
    StructField("order_id", ShortType(), nullable=False),
    StructField("product_id", ShortType(), nullable=False),
    StructField("unit_price", FloatType(), nullable=False),
    StructField("quantity", ShortType(), nullable=False),
    StructField("discount", FloatType(), nullable=False),
])

In [ ]:
# Lendo CSV com schema explicito -- sem inferencia
order_details_csv = spark.read.csv(
    f"{VOLUME_PATH}/order_details_csv",
    header=True,
    schema=order_details_schema,
)

order_details_csv.printSchema()

In [ ]:
# Comparando: schema inferido vs schema explicito
order_details_inferido = spark.read.csv(
    f"{VOLUME_PATH}/order_details_csv",
    header=True,
    inferSchema=True,
)

print("=== Schema INFERIDO ===")
order_details_inferido.printSchema()

print("=== Schema EXPLICITO ===")
order_details_csv.printSchema()

> **Por que isso importa?** Veja as diferencas que o `inferSchema` introduziu:
>
> | Coluna | Inferido | Explicito | Diferenca |
> |---|---|---|---|
> | `order_id`, `product_id`, `quantity` | `integer` (32 bits) | `short` (16 bits) | 4 bytes vs 2 bytes por valor |
> | `unit_price`, `discount` | `double` (64 bits) | `float` (32 bits) | 8 bytes vs 4 bytes por valor |
>
> O `inferSchema` sempre "chuta pra cima" — usa `integer` em vez de `short` e `double` em vez de `float` porque, olhando so o CSV, ele nao tem como saber o range real dos dados.
>
> **Para o Northwind (830 linhas), nao faz diferenca perceptivel.** Mas em producao com bilhoes de linhas, cada byte conta:
> - **Memoria** — `short` ocupa metade do `integer`. Em 1 bilhao de linhas, sao ~2 GB a menos so nas colunas de ID
> - **Disco** — arquivos menores = leitura mais rapida do storage
> - **Shuffle** — menos dados trafegando entre executors em `join`/`groupBy`
>
> Alem do tamanho, tem a questao de **correcao semantica**: `float` tem ~7 digitos de precisao e `double` tem ~15. Para preco unitario, `float` basta. Para calculos financeiros de alta precisao, `double` ou `decimal` seria melhor.
>
> **Regra pratica:** em producao, sempre defina o schema explicitamente. Voce ganha tipo correto, menos memoria e leitura mais rapida (sem a passada extra do `inferSchema`).

## 5. `spark.table()` vs `spark.read` -- quando usar cada um

| Metodo | Quando usar | Exemplo |
|--------|-------------|----------|
| `spark.table("tabela")` | Tabela ja registrada no Unity Catalog | `spark.table("orders")` |
| `spark.read.format().load()` | Arquivos em volumes, DBFS ou storage externo | `spark.read.csv("/path/file.csv")` |

**Regra pratica:** Se a tabela esta no catalogo, use `spark.table()`. Se voce esta lendo arquivos brutos (CSV, JSON, Parquet), use `spark.read`.

In [ ]:
# Abordagem 1: spark.table() -- tabela do catalogo
products = spark.table("products")
display(products.limit(5))

In [ ]:
# Abordagem 2: spark.read -- arquivo CSV
# (usando o CSV que gravamos antes)
orders_from_file = spark.read.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,
    inferSchema=True,
)
display(orders_from_file.limit(5))

## 6. Formatos de leitura

O PySpark suporta varios formatos de leitura. Vamos demonstrar os principais.

### 6.1 CSV

Formato texto, separado por virgulas. Simples, mas sem informacao de tipos -- precisa de `inferSchema` ou schema explicito.

In [ ]:
# Ja demonstrado acima. Opcoes comuns:
df_csv = spark.read.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,        # primeira linha e cabecalho
    inferSchema=True,   # inferir tipos (ou usar schema=)
    sep=",",            # separador (padrao e virgula)
    nullValue="NA",     # tratar "NA" como nulo
)
print(f"CSV: {df_csv.count()} linhas")

### 6.2 JSON

In [ ]:
# Escrevendo orders como JSON
orders.write.json(
    f"{VOLUME_PATH}/orders_json",
    mode="overwrite",
)

# Lendo de volta
df_json = spark.read.json(f"{VOLUME_PATH}/orders_json")
print(f"JSON: {df_json.count()} linhas")
df_json.printSchema()

### 6.3 Parquet

Formato **colunar** e binario. Muito mais eficiente que CSV/JSON porque:
- Armazena tipos nativamente (sem necessidade de inferSchema)
- Compressao por coluna
- Leitura seletiva de colunas (column pruning)

In [ ]:
# Escrevendo orders como Parquet
orders.write.parquet(
    f"{VOLUME_PATH}/orders_parquet",
    mode="overwrite",
)

# Lendo de volta -- note que nao precisa de inferSchema
df_parquet = spark.read.parquet(f"{VOLUME_PATH}/orders_parquet")
print(f"Parquet: {df_parquet.count()} linhas")
df_parquet.printSchema()

### 6.4 Delta

Formato nativo do Databricks. Parquet + log de transacoes. Suporte a ACID, time travel, schema evolution. E o formato que voce ja conhece das aulas anteriores.

In [ ]:
# Escrevendo orders como Delta
orders.write.format("delta").mode("overwrite").save(
    f"{VOLUME_PATH}/orders_delta"
)

# Lendo de volta
df_delta = spark.read.format("delta").load(f"{VOLUME_PATH}/orders_delta")
print(f"Delta: {df_delta.count()} linhas")
df_delta.printSchema()

### Comparacao de formatos

| Formato | Tipos nativos | Compressao | Schema obrigatorio | Transacional | Uso recomendado |
|---------|:---:|:---:|:---:|:---:|---|
| CSV     | Nao | Nao | Nao | Nao | Importacao/exportacao simples |
| JSON    | Parcial | Nao | Nao | Nao | APIs, dados semi-estruturados |
| Parquet | Sim | Sim | Nao | Nao | Data lakes, consultas analiticas |
| Delta   | Sim | Sim | Nao | Sim | **Padrao no Databricks** -- sempre que possivel |

## 7. Exploracao de DataFrames

Agora que sabemos carregar dados, vamos explorar os DataFrames. Essas funcoes sao essenciais para entender a estrutura e o conteudo dos dados antes de qualquer transformacao.

### 7.1 `display(df)` -- Visualizacao nativa do Databricks

Funcao exclusiva do Databricks. Mostra os dados em formato tabular interativo com opcoes de grafico.

In [ ]:
display(orders)

### 7.2 `df.printSchema()` -- Arvore de tipos

Mostra a estrutura do DataFrame em formato de arvore, com nome e tipo de cada coluna.

In [ ]:
orders.printSchema()

In [ ]:
customers.printSchema()

### 7.3 `df.describe()` -- Estatisticas descritivas

Retorna contagem, media, desvio padrao, minimo e maximo para colunas numericas.

In [ ]:
display(orders.describe())

### 7.4 `df.dtypes` -- Lista de tuplas (nome, tipo)

In [ ]:
# Retorna uma lista de tuplas (nome_coluna, tipo)
orders.dtypes

### 7.5 `df.columns` -- Lista de nomes das colunas

In [ ]:
# Retorna uma lista Python com os nomes das colunas
print("Colunas de orders:", orders.columns)
print(f"Total: {len(orders.columns)} colunas")

print("\nColunas de customers:", customers.columns)
print(f"Total: {len(customers.columns)} colunas")

### 7.6 `df.count()` -- Contagem de linhas

In [ ]:
# count() e uma ACTION -- dispara a execucao no cluster
print(f"orders: {orders.count()} registros")
print(f"customers: {customers.count()} registros")
print(f"order_details: {order_details.count()} registros")
print(f"products: {products.count()} registros")

### 7.7 `df.show()` -- Exibicao no console

Diferente do `display()`, o `show()` imprime no console como texto. Util para debugging e funciona fora do Databricks tambem.

Parametros:
- `n`: numero de linhas (padrao 20)
- `truncate`: truncar colunas longas (padrao True, 20 caracteres)

In [ ]:
# show() com parametros
orders.show(5, truncate=False)

In [ ]:
# show() com truncate padrao -- compare a diferenca
customers.show(5)

## 8. DataFrame API vs `spark.sql()` -- duas formas de fazer a mesma coisa

O PySpark oferece duas formas equivalentes de consultar dados:

1. **DataFrame API** -- encadeamento de metodos Python (`.filter()`, `.select()`, etc.)
2. **`spark.sql()`** -- escrever SQL puro como string

Ambas produzem o **mesmo plano de execucao**. A escolha e questao de estilo e contexto. Nesta aula, vamos focar na DataFrame API, mas sempre mostrando o equivalente SQL como referencia.

In [ ]:
# Para usar spark.sql(), primeiro criamos uma temp view
products.createOrReplaceTempView("products_view")

In [ ]:
# DataFrame API: produtos com preco > 50
resultado_df = (
    products
    .filter(F.col("unit_price") > 50)
    .select("product_name", "unit_price")
    .orderBy(F.col("unit_price").desc())
)

display(resultado_df)

In [ ]:
# spark.sql(): mesma consulta em SQL puro
resultado_sql = spark.sql("""
    SELECT product_name, unit_price
    FROM products_view
    WHERE unit_price > 50
    ORDER BY unit_price DESC
""")

display(resultado_sql)

> **Resultado identico!** As duas abordagens geram o mesmo plano de execucao no Spark. A DataFrame API e mais "Pythonica" e permite composicao programatica. O `spark.sql()` e mais familiar para quem vem do SQL. Nesta trilha, vamos priorizar a DataFrame API -- que e o objetivo desta aula.

---

## 9. Conceitos de Arquitetura Spark (Prova)

Estes conceitos aparecem no exame Databricks Certified Associate Developer for Apache Spark. Vamos demonstrar os principais.

### 9.1 Lazy Evaluation — Transformações vs Ações

O Spark usa **lazy evaluation**: transformações (`select`, `filter`, `withColumn`, `join`) NÃO são executadas imediatamente. Elas apenas constroem um **plano de execução** (DAG).

A execução só acontece quando uma **ação** é chamada:

| Transformações (lazy) | Ações (executam) |
|---|---|
| `select()`, `filter()`, `withColumn()` | `show()`, `count()`, `collect()` |
| `join()`, `groupBy()`, `orderBy()` | `display()`, `write`, `take()` |

Isso permite ao Spark **otimizar** o plano inteiro antes de executar (ex: Catalyst Optimizer).

### Como visualizar o plano?

Use `df.explain(True)` para ver as 4 etapas do plano:

1. **Parsed Logical Plan** — o que voce escreveu, traduzido para o Spark
2. **Analyzed Logical Plan** — com tipos e nomes de colunas resolvidos
3. **Optimized Logical Plan** — apos otimizacoes do Catalyst (ex: predicate pushdown, column pruning)
4. **Physical Plan** — o plano real que sera executado no cluster (scans, shuffles, sorts)

In [ ]:
# Lazy evaluation: nenhuma dessas linhas executa nada no cluster
transformed = (
    orders
    .filter(F.year(F.col("order_date")) == 1997)
    .select("order_id", "customer_id", "order_date")
    .orderBy("order_date")
)

print(type(transformed))  # Apenas um DataFrame com plano montado

# explain() mostra o plano que o Spark MONTOU mas ainda NAO executou
# Ele revela as otimizacoes do Catalyst Optimizer
print("\n=== Plano de execucao (explain) ===")
transformed.explain(True)

# Somente agora o Spark executa (count é uma AÇÃO)
print(f"Total: {transformed.count()} registros")

### 9.2 Caching e Persist (conceitual)

Quando voce reutiliza um DataFrame varias vezes, pode **cachea-lo** na memoria para evitar recomputacao.

- `df.cache()` — armazena na memoria (equivalente a `MEMORY_AND_DISK`)
- `df.persist(StorageLevel)` — controle fino do nivel de armazenamento
- `df.unpersist()` — libera o cache

> **Nota:** `cache()` e `persist()` **nao funcionam no Serverless compute** (versao gratuita do Databricks). O codigo abaixo e apenas conceitual — para executar, voce precisaria de um cluster classico.

In [ ]:
# ⚠️ NAO EXECUTE no Serverless — apenas conceitual
# Em um cluster classico, o codigo seria:

# orders_cached = orders.cache()
# print(f"Contagem (1a vez - computa e cacheia): {orders_cached.count()}")
# print(f"Contagem (2a vez - le do cache): {orders_cached.count()}")
# print(f"Esta cacheado? {orders_cached.is_cached}")
# orders_cached.unpersist()

print("cache() e persist() nao sao suportados no Serverless compute.")
print("Em cluster classico, cache() evita recomputacao de DataFrames reutilizados.")

### 9.3 Hierarquia de Execucao: Job → Stage → Task

Quando voce chama uma **acao** (`count()`, `show()`, `collect()`), o Spark cria uma hierarquia de execucao:

```
Acao (ex: count())
  └── Job (1 job por acao)
        └── Stage (separado por shuffles — groupBy, join, orderBy)
              └── Task (1 task por particao de dados)
```

**Exemplo concreto:** imagine `orders.groupBy("country").count().show()`

1. **Job 1** — disparado pelo `show()`
2. **Stage 1** — Scan da tabela orders (leitura das particoes)
3. **Stage 2** — Shuffle para agrupar por country (redistribui dados entre executors)
4. **Tasks** — se `orders` tem 8 particoes, o Stage 1 tera 8 tasks paralelas

> Voce pode ver essa hierarquia no **Spark UI** (aba "Jobs" e "Stages") clicando em "See performance" no output de cada celula.

### 9.4 `collect()` e `row.asDict()` — DataFrame → Python

O `collect()` e uma **acao** que traz todos os dados do cluster para o Driver como uma **lista de objetos Row**.

**Cuidado:** se o DataFrame tem milhoes de linhas, `collect()` pode estourar a memoria do Driver. Use `limit()` antes ou prefira `take(n)`.

```python
# DataFrame (distribuido no cluster) → lista Python (local no Driver)
rows = df.collect()        # lista de Row
dicts = [r.asDict() for r in rows]  # lista de dict
```

In [ ]:
# Passo 1: collect() traz os dados do cluster para o Driver
top_5 = (
    products
    .select("product_name", "unit_price")
    .orderBy(F.col("unit_price").desc())
    .limit(5)
    .collect()  # ACAO — retorna lista de Row
)

# Passo 2: cada elemento e um objeto Row
print("Tipo da lista:", type(top_5))
print("Tipo de cada elemento:", type(top_5[0]))
print("Primeiro Row:", top_5[0])

# Passo 3: converter Row → dict com asDict()
print("\n--- Convertendo para dicionarios ---")
lista_dicts = [row.asDict() for row in top_5]
for item in lista_dicts:
    print(f"  {item['product_name']}: ${item['unit_price']}")

# Passo 4: agora e Python puro — pode usar em qualquer lugar
print(f"\nProduto mais caro: {lista_dicts[0]['product_name']}")
print(f"Tipo final: {type(lista_dicts[0])}")

#### Questao 4 — Sort, printSchema e conversao para lista

> Ordenar `employeeDF` por salary e age DESC, imprimir schema e converter para lista de rows. Qual codigo esta correto?
>
> - **A)** `toPandas().values.toList()` — funciona mas usa `desc("salary")` com sintaxe errada
> - **B)** `schema()` nao existe (e `printSchema()`) e `toList()` nao e metodo do DataFrame
> - **C) `orderBy(col("salary").desc(), col("age").desc())` + `.collect()` + `printSchema()` + `row.asDict()`** ✅
> - **D)** `descending=True` nao e parametro valido, `describe()` nao e `printSchema()`, `show()` retorna None
>
> **Resposta: C** — A forma correta de converter DataFrame → lista Python e: `.collect()` retorna lista de `Row`, e `row.asDict()` converte cada Row em dicionario.

### 9.4 Deployment Modes

O Spark pode rodar em tres modos de deploy:

| Modo | Driver roda em | Executors rodam em | Uso tipico |
|------|---------------|-------------------|------------|
| **Local** | Maquina local (1 JVM) | Mesma maquina | Desenvolvimento, testes |
| **Client** | Maquina do usuario | Workers do cluster | Debug interativo (notebooks) |
| **Cluster** | Dentro do cluster | Workers do cluster | Producao |

> **Na prova:** Questao 7 — "Local mode" e o unico onde tudo roda em uma unica maquina/JVM.

#### Questao 7 — Deployment mode com single node

> Um engenheiro precisa rodar Spark de forma que **todos os executors rodem em um unico worker node**. Qual modo de deploy ele deve usar?
>
> - **A)** Cluster mode — driver roda no cluster, executors em multiplos workers
> - **B) Local mode — tudo roda em uma unica JVM na mesma maquina** ✅
> - **C)** Client mode — driver na maquina cliente, executors distribuidos no cluster
> - **D)** Standard mode — nao existe esse modo no Spark
>
> **Resposta: B** — Local mode e o unico onde Driver + Executors rodam na mesma maquina/JVM.

In [ ]:
# Verificar config atual de shuffle partitions
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Exemplo: alterar para 8 (útil para datasets pequenos como Northwind)
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("Novo valor:", spark.conf.get("spark.sql.shuffle.partitions"))

# Voltar ao padrão
spark.conf.set("spark.sql.shuffle.partitions", "200")

#### Questao 5 — spark.sql.shuffle.partitions = 200

> Qual o impacto de configurar `spark.sql.shuffle.partitions = 200`?
>
> - **A) DataFrames serao divididos em 200 particoes durante operacoes de shuffle** ✅
> - **B)** Aloca memoria de 200 executors — nao tem relacao, e sobre particoes
> - **C)** Todos os DataFrames terao 200 particoes — nao, so os que passam por shuffle
> - **D)** Spark processa apenas as primeiras 200 particoes — nao, todas sao processadas
>
> **Resposta: A** — Esta config so afeta operacoes de shuffle (`groupBy`, `join`, `orderBy`). DataFrames que nao passam por shuffle mantem suas particoes originais.

---

## Exercicios

Agora e sua vez! Tente resolver os exercicios abaixo antes de olhar as solucoes.

### Exercicio 1

Carregue a tabela `customers` e mostre:
1. O schema (`printSchema()`)
2. A contagem total de registros
3. Os 10 primeiros registros (`show()` com `truncate=False`)

In [ ]:
# Exercicio 1 -- sua solucao aqui

In [ ]:
# Exercicio 1 -- Solucao
customers = spark.table("customers")

# 1. Schema
print("=== Schema ===")
customers.printSchema()

# 2. Contagem
print(f"\nTotal de clientes: {customers.count()}")

# 3. Primeiros 10 registros
print("\n=== 10 primeiros registros ===")
customers.show(10, truncate=False)

### Exercicio 2

Carregue a tabela `products`:
1. Escreva como CSV em `{VOLUME_PATH}/products_csv`
2. Leia o CSV com `inferSchema=True` e exiba o schema
3. Defina um schema explicito com `StructType` e leia novamente
4. Compare os dois schemas -- quais diferencas voce encontra?

**Dica:** As colunas de `products` sao: `product_id` (SMALLINT), `product_name` (STRING), `supplier_id` (SMALLINT), `category_id` (SMALLINT), `quantity_per_unit` (STRING), `unit_price` (FLOAT), `units_in_stock` (SMALLINT), `units_on_order` (SMALLINT), `reorder_level` (SMALLINT), `discontinued` (INT)

In [ ]:
# Exercicio 2 -- sua solucao aqui

In [ ]:
# Exercicio 2 -- Solucao

# 1. Escrever products como CSV
products = spark.table("products")
products.write.csv(
    f"{VOLUME_PATH}/products_csv",
    header=True,
    mode="overwrite",
)

# 2. Ler com inferSchema
products_inferido = spark.read.csv(
    f"{VOLUME_PATH}/products_csv",
    header=True,
    inferSchema=True,
)
print("=== Schema INFERIDO ===")
products_inferido.printSchema()

# 3. Definir schema explicito
products_schema = StructType([
    StructField("product_id", ShortType(), False),
    StructField("product_name", StringType(), False),
    StructField("supplier_id", ShortType(), True),
    StructField("category_id", ShortType(), True),
    StructField("quantity_per_unit", StringType(), True),
    StructField("unit_price", FloatType(), True),
    StructField("units_in_stock", ShortType(), True),
    StructField("units_on_order", ShortType(), True),
    StructField("reorder_level", ShortType(), True),
    StructField("discontinued", IntegerType(), False),
])

products_explicito = spark.read.csv(
    f"{VOLUME_PATH}/products_csv",
    header=True,
    schema=products_schema,
)
print("=== Schema EXPLICITO ===")
products_explicito.printSchema()

### Exercicio 3

Escreva a mesma consulta de duas formas:

**Consulta:** Listar `first_name`, `last_name` e `hire_date` dos employees contratados depois de 1993-01-01, ordenados por data de contratacao.

1. Usando DataFrame API (`.filter()`, `.select()`, `.orderBy()`)
2. Usando `spark.sql()` (crie uma temp view primeiro)

In [ ]:
# Exercicio 3 -- sua solucao aqui

In [ ]:
# Exercicio 3 -- Solucao

employees = spark.table("employees")

# Abordagem 1: DataFrame API
print("=== DataFrame API ===")
resultado_api = (
    employees
    .filter(F.col("hire_date") > "1993-01-01")
    .select("first_name", "last_name", "hire_date")
    .orderBy("hire_date")
)
resultado_api.show(truncate=False)

# Abordagem 2: spark.sql()
print("=== spark.sql() ===")
employees.createOrReplaceTempView("employees_view")

resultado_sql = spark.sql("""
    SELECT first_name, last_name, hire_date
    FROM employees_view
    WHERE hire_date > '1993-01-01'
    ORDER BY hire_date
""")
resultado_sql.show(truncate=False)

---

## Resumo do Bloco 01

Neste bloco voce aprendeu:

| Conceito | Funcao/Metodo |
|----------|---------------|
| Carregar tabela do catalogo | `spark.table("tabela")` |
| Ler arquivo CSV | `spark.read.csv(path, header=True, inferSchema=True)` |
| Ler arquivo JSON | `spark.read.json(path)` |
| Ler arquivo Parquet | `spark.read.parquet(path)` |
| Ler arquivo Delta | `spark.read.format("delta").load(path)` |
| Schema explicito | `StructType([StructField("col", Tipo(), nullable)])` |
| Visualizar dados | `display(df)`, `df.show(n, truncate=False)` |
| Inspecionar schema | `df.printSchema()`, `df.dtypes` |
| Listar colunas | `df.columns` |
| Contar linhas | `df.count()` |
| Estatisticas | `df.describe()` |
| SQL direto | `spark.sql("SELECT ...")` |
| Criar temp view | `df.createOrReplaceTempView("nome")` |

No proximo bloco, vamos aprender **transformacoes**: `select()`, `filter()`, `withColumn()`, funcoes de string, data e tratamento de nulos.